In [ ]:
import os
import sys
from datetime import datetime, timedelta
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

API_KEY = os.getenv("YOUTUBE_TOKEN")      # Apikey generada en google youtube
if not API_KEY:
    print("Error: no se encontro YOUTUBE_TOKEN en .env")
    sys.exit(1)

CANAL_ID = "xxxxxxxxxxxxxx"     # Localizar el id del canal y reemplazar

# Rango de fechas. 
FECHA_DESDE = "2026-07-01"
FECHA_HASTA = "2026-09-15"

desde_dt = datetime.strptime(FECHA_DESDE, "%Y-%m-%d")
hasta_dt = datetime.strptime(FECHA_HASTA, "%Y-%m-%d")

if hasta_dt < desde_dt:
    print("Error: FECHA_HASTA es anterior a FECHA_DESDE")
    sys.exit(1)

INICIO = f"{FECHA_DESDE}T03:00:00Z"
FIN = f"{(hasta_dt + timedelta(days=1)).strftime('%Y-%m-%d')}T03:00:00Z"

SALIDA = f"comentarios_{FECHA_DESDE}_a_{FECHA_HASTA}.csv"
SEP = "|"

youtube = build("youtube", "v3", developerKey=API_KEY)

In [ ]:
respuesta = youtube.channels().list(
    part="contentDetails,status",
    id=CANAL_ID
).execute()

if not respuesta.get("items"):
    print("No se encontro el canal")
    sys.exit(1)

canal = respuesta["items"][0]
uploads = canal["contentDetails"]["relatedPlaylists"]["uploads"]
made_for_kids = canal.get("status", {}).get("madeForKids", False)

print(f"Made for Kids: {made_for_kids}")

In [ ]:
videos = []
pagina = None

while True:
    r = youtube.playlistItems().list(
        part="snippet,contentDetails",
        playlistId=uploads,
        maxResults=50,
        pageToken=pagina
    ).execute()

    for item in r.get("items", []):
        pub = item["snippet"]["publishedAt"]
        if INICIO <= pub < FIN:
            videos.append(item["contentDetails"]["videoId"])

    pagina = r.get("nextPageToken")
    if not pagina:
        break

print(f"Videos encontrados: {len(videos)}")
for v in videos:
    print(v)

In [ ]:
def limpiar(texto):
    texto = texto.replace("\r", " ").replace("\n", " ").strip()
    texto = texto.replace(SEP, " ")
    return texto


def descargar(video_id, archivo):
    pagina = None
    total = 0
    while True:
        try:
            r = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                pageToken=pagina,
                textFormat="plainText"
            ).execute()
        except HttpError as e:
            contenido = e.content.decode("utf-8", errors="ignore")
            if "commentsDisabled" in contenido:
                return total, "COMENTARIOS_DESHABILITADOS"
            if "quotaExceeded" in contenido:
                return total, "CUOTA_EXCEDIDA"
            return total, "ERROR_403"

        items = r.get("items", [])
        for item in items:
            c = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
            c = limpiar(c)
            if c:
                archivo.write(c + SEP + "\n")
                total += 1

        archivo.flush()
        pagina = r.get("nextPageToken")
        if not pagina:
            break

    return total, "OK"

In [ ]:
total = 0

with open(SALIDA, "w", encoding="utf-8", newline="\n") as f:
    f.write("comentario" + SEP + "sentimiento\n")
    for i, vid in enumerate(videos, start=1):
        print(f"[{i}/{len(videos)}] {vid}")
        try:
            c, estado = descargar(vid, f)
        except HttpError:
            c, estado = 0, "ERROR_HTTP"
        print(f"   {estado}: {c} comentarios")
        if estado == "CUOTA_EXCEDIDA":
            break
        total += c

print(f"\nTotal: {total} comentarios")
print(f"Archivo: {SALIDA}")

In [ ]:
if os.path.exists(SALIDA):
    with open(SALIDA, "r", encoding="utf-8") as f:
        lineas = f.readlines()
    print(f"Lineas: {len(lineas)}")
    print("--- Primeras 10 ---")
    for l in lineas[:10]:
        print(l.rstrip())